Example 1 

In [ ]:
import numpy as np 
from time import time # for comparing time 
d,N = 1000,10000 
X = np.random.randn(N,d)
z = np.random.randn(d)

def dist_pp(z,x):
    d = z- x.reshape(z.shape)
    return np.sum(d*d)

def dist_ps_naive (z,X):
    N = X.shape[0]
    res = np.zeros((1,N))
    for i in range(N):
        res[0][i] = dist_pp(z,X[i])
    return res

def dist_ps_fast(z,X):
    X2 = np.sum(X*X,1) #X*X is multiplying each element with itself, not matrix multiplication 
    z2 = np.sum(z*z)
    return X2 + z2 -2*X.dot(z)

t1 = time() 
D1 = dist_ps_naive(z,X)
print('Naive point2set, running time:', time() - t1, "s")

t1 = time()
D2 = dist_ps_fast(z,X)
print("Fast point2set, running time:", time() -t1, "s")
print("Result difference:", np.linalg.norm(D1-D2))




Naive point2set, running time: 0.06179070472717285 s
Fast point2set, running time: 0.044069766998291016 s
Result difference: 1.891306205089223e-11


Example 2

In [ ]:
Z = np.random.randn(100,d)

def dist_ss_0(Z,X):
    M,N = Z.shape[0], X.shape[0]
    res = np.zeros((M,N))
    for i in range(M):
        res[i] = dist_ps_fast(Z[i],X)
    return res 

def dist_ss_fast(Z,X):
    X2 = np.sum(X*X,1) # summary of each square elemnt on a row 
    Z2 = np.sum(Z*Z,1)
    return Z2.reshape(-1,1) + X2.reshape(1,-1) - 2*Z.dot(X.T) #(-1,1) means (n,1) for n is calculated based on the number of elements

t1 = time()
D3 = dist_ss_0(Z,X)
print("Half fast set2set running time:", time() - t1, "s")

t1 = time()
D4 = dist_ss_fast(Z,X)
print("Fast set2set running time:", time() -t1, "s")

print("Result difference:", np.linalg.norm(D3-D4))


Half fast set2set running time: 4.635662794113159 s
Fast set2set running time: 0.09250807762145996 s
Result difference: 1.0448655376269507e-10


Example 3: Iris Database 

In [ ]:
import numpy as np 
from sklearn import neighbors, datasets
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score 

iris = datasets.load_iris()
iris_X = iris.data 
iris_y = iris.target 

print("Labels:", np.unique(iris_y))

#split train and test 
np.random.seed(7)
X_train, X_test, y_train, y_test = train_test_split(iris_X, iris_y, test_size = 130)
print("Training size:", X_train.shape[0], ", test size:", X_test.shape[0])

# 1NN
model_1 = neighbors.KNeighborsClassifier(n_neighbors = 1, p = 2) #p = 2 means l2 norm distance 
model_1.fit(X_train, y_train)
y_pred = model_1.predict(X_test)
print("Accuracy of 1NN:%.2f%%"% (100*accuracy_score(y_test, y_pred)))

#7NN
model_7 = neighbors.KNeighborsClassifier(n_neighbors = 7, p = 2)
model_7.fit(X_train, y_train)
y_pred_1 = model_7.predict(X_test)
print("Accuracy of 7NN with major voting: %.2f%%"%(100*accuracy_score(y_test, y_pred_1)))


Labels: [0 1 2]
Training size: 20 , test size: 130
Accuracy of 1NN:92.31%
Accuracy of 7NN with major voting: 93.85%


In [17]:
# 7NN with weights = 'distance'
model_7 = neighbors.KNeighborsClassifier(n_neighbors = 7, p = 2, weights = 'distance')
model_7.fit(X_train, y_train)
y_pred_1 = model_7.predict(X_test)
print("Accuracy of 7NN (1/distance weights): %.2f%%"%(100*accuracy_score(y_test, y_pred_1)))

Accuracy of 7NN (1/distance weights): 94.62%
